# Executing prompts programmatically
- generated completions by LLM applications rely on well-crafted prompts
- LangChain prompts are typically constructed and sent to the LLM programmatically as part of a larger workflow.

we’ll start by setting up and executing prompts using the Ollama API directly, and then explore how to run and manage prompts within LangChain.

In [2]:
from langchain_ollama import OllamaLLM , ChatOllama

LLM = OllamaLLM(model="qwen3-coder-next:cloud")
chat = ChatOllama(model="qwen3-coder-next:cloud")


## Start Prompts with Chat Model from ollama in langchain

In [4]:
prompt_input = """Write a coincise message to remind 
users to be vigilant about phishing attacks."""
response = chat.invoke([
    {"role": "system", 
     "content": "You are a helpful assistant."},
    {"role": "user", 
     "content": prompt_input}])

 
print(response)

content='**Phishing Reminder**  \nAlways stay alert: phishing attacks often disguise as trusted sources (email, SMS, calls) to steal your info.  \n✔️ Double-check sender addresses  \n✔️ Never click suspicious links or download unknown attachments  \n✔️ Verify requests for sensitive info through official channels  \nWhen in doubt—don’t click. Report and delete. Stay secure!' additional_kwargs={} response_metadata={'model': 'qwen3-coder-next:cloud', 'created_at': '2026-02-28T09:45:12.466252298Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3306173548, 'load_duration': None, 'prompt_eval_count': 35, 'prompt_eval_duration': None, 'eval_count': 75, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'} id='lc_run--019ca3a3-377c-7633-9d58-e38f647187a7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 35, 'output_tokens': 75, 'total_tokens': 110}


In [11]:
print(f"structured_response\n\tcontent: {response.content}\n\n\tmetadata: {response.response_metadata}\n\n\tadditional_kwargs {response.additional_kwargs}") 

structured_response
	content: **Phishing Reminder**  
Always stay alert: phishing attacks often disguise as trusted sources (email, SMS, calls) to steal your info.  
✔️ Double-check sender addresses  
✔️ Never click suspicious links or download unknown attachments  
✔️ Verify requests for sensitive info through official channels  
When in doubt—don’t click. Report and delete. Stay secure!

	metadata: {'model': 'qwen3-coder-next:cloud', 'created_at': '2026-02-28T09:45:12.466252298Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3306173548, 'load_duration': None, 'prompt_eval_count': 35, 'prompt_eval_duration': None, 'eval_count': 75, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3-coder-next:cloud', 'model_provider': 'ollama'}

	additional_kwargs {}


## Prompt templates
- creating flexible prompt templates that incorporate user input through parameters is crucial.
- `PromptTemplate` class, lets you manage and reuse parametrized prompts without the need for custom functions
- creation of dynamic prompts
- enhances the efficiency and adaptability of your LLM interactions


In [13]:
from langchain_core.prompts import PromptTemplate
 
prompt_template = PromptTemplate.from_template(
"""You are an experienced copywriter. 
Write a {num_words} words summary the the following text, 
using a {tone} tone: {text}""")

In [14]:
segovia_aqueduct_text = """The Aqueduct of Segovia (Spanish: Acueducto de Segovia) is a Roman aqueduct in Segovia, Spain. It was built around the first century AD to channel water from springs in the mountains 17 kilometres (11 mi) away to the city's fountains, 
public baths and private houses, and was in use until 1973. Its elevated section, with its complete arcade of 167 arches, is one of the best-preserved Roman aqueduct bridges and the foremost symbol of Segovia, as evidenced by its presence on the city's coat of arms. 
The Old Town of Segovia and the aqueduct, were declared a UNESCO World Heritage Site in 1985. As the aqueduct lacks a legible inscription (one was apparently located in the structure's attic, or top portion[citation needed]), 
the date of construction cannot be definitively determined. The general date of the Aqueduct's construction was long a mystery, although it was thought to have been during the 1st century AD, during the reigns of the Emperors Domitian, 
Nerva, and Trajan. At the end of the 20th century, Géza Alföldy deciphered the text on the dedication plaque by studying the anchors that held the now missing bronze letters in place. He determined that Emperor Domitian (AD 81–96) ordered 
its construction[1] and the year 98 AD was proposed as the most likely date of completion.[2] However, in 2016 archeological evidence was published which points to a slightly later date, after 112 AD, during the government of Trajan or in the beginning of 
the government of emperor Hadrian, from 117 AD."""

In [15]:
prompt = prompt_template.format(
    text=segovia_aqueduct_text,
    num_words=20, 
    tone="knowledgeable and engaging")

In [17]:
response = LLM.invoke(prompt)
print(response)

Built c. 98 AD under Domitian (later revised to post-112 AD under Trajan/Hadrian), Segovia’s majestic 167-arch Roman aqueduct delivered mountain water for 1,900 years—now a UNESCO World Heritage symbol and Spain’s best-preserved aqueduct bridge.


### Types of prompt
#### 1. Text classification

In [19]:
Instruction = "Classify the following text into one of these categories: history, tech, gardening."
example = """Headphones provide immersive audio experiences for music lovers and gamers alike. 
            The text "Headphones provide immersive audio experiences for music lovers and gamers alike." should be classified as "tech" because it discusses technology-related products and their functionality."""

classification_template = PromptTemplate.from_template(
"""instruction: {instruction},\nexample: {example}\n Text: {text} Category is **only the category""")

prompt = classification_template.format(
    instruction=Instruction,
    example=example, 
    text="Headphones provide immersive audio experiences for music lovers and gamers alike.")

response = LLM.invoke(prompt)
response

'tech'

In [20]:
history_text = """Julius Caesar was a Roman general and statesman who played a pivotal role in the demise of the Roman Republic and the rise of the Roman Empire"""
prompt = classification_template.format(
    instruction=Instruction,
    example=example, 
    text=history_text)

response = LLM.invoke(prompt)
response

'history'

#### 2. Sentiment analysis



In [ ]:
Instruction = """ Instruction: Classify the following text as positive, neutral or negative"""


sentiment_template = PromptTemplate.from_template(
"""instruction: {instruction},\ntext: {text} """)

prompt = sentiment_template.format(
    instruction=Instruction,
    text="Apple have announced stellar quarterly results")

response = LLM.invoke(prompt)
response

'Positive'

In [22]:
prompt = sentiment_template.format(
    instruction=Instruction,
    text="this is the worst movie I have watched this month")

response = LLM.invoke(prompt)
response

'Negative'

In [23]:
prompt = sentiment_template.format(
    instruction=Instruction,
    text="The customer bought a green dress")

response = LLM.invoke(prompt)
response

'Neutral'

In [28]:
instruction = """Instruction: classify the sentiment of following stock reports as positive, neutral or negative"""

stock_report = """ Stock 1: Apple: the launch of the new iPhone has been a success
Stock 2: Nvidia: sales are being propelled by consumer demand on LLMs
Stock 3: GX oil: demand of carbon based energy is dropping thanks to the increase of renewable energy
Output: output a table with the columns "stock name", "sentiment" """

prompt = sentiment_template.format(
    instruction=instruction,
    text=stock_report)

response = LLM.invoke(prompt)
display(response)

'| Stock Name | Sentiment |\n|------------|-----------|\n| Apple      | Positive  |\n| Nvidia     | Positive  |\n| GX oil     | Negative  |'

#### 3. Text summarization

In [29]:
instruction = """write a 30 word summary for the following text"""

text = """ Home PCs from the 1980s were iconic pioneers of personal computing. 
                    Computers like the Apple II, IBM PC, and Commodore 64 revolutionized how we lived and worked.
                    They introduced us to word processing, gaming, and the possibilities of home computing. The graphical user interface, 
                    as seen in the Apple Macintosh, changed the way we interacted with machines. These PCs, though less powerful by today's standards, 
                    ignited a digital revolution, shaping the technology landscape we know today. They hold a special place in history, 
                    symbolizing the dawn of a new era in computing that laid the foundation for the modern digital age. """


template = PromptTemplate.from_template(
"""instruction: {instruction},\ntext: {text} """)

prompt = template.format(
    instruction=instruction,
    text=text)

response = LLM.invoke(prompt)
display(response)

'1980s home PCs—Apple II, IBM PC, Commodore 64, Macintosh—pioneered personal computing, introducing word processing, gaming, GUIs, and sparking the digital revolution that shaped today’s tech landscape.'

#### 4. Composing text

In [34]:
instruction = """Write a piece on the diver watches, mentioning the following facts:"""
fact = """- the Rolex Oyster Perpetual was among the first to be water resistant
- Omega produced the Seamaster in 1948 as the first purpose designed diver watch was the Seamaster
= the Seamaster set a world record of immersion of 62.5m in 1955.
= Rolex produced the first Submariner in 1953
= the Submariner became popular also because of the association with James Bond until 1995
= James Bond started to wear the Seamaster in 1995"""

prompt = template.format(
    instruction=instruction,
    text=fact)

response = LLM.invoke(prompt)
display(response)


'Diver watches owe their origins to two Swiss legends: Rolex and Omega. Rolex’s Oyster Perpetual was among the first to boast real water resistance, paving the way for the Submariner in 1953—the first dive watch built from the ground up for professional use. Around the same time, Omega introduced the Seamaster in 1948, often credited as the first purpose-built diver. In 1955, a Seamaster even dived to 62.5 meters, setting a world immersion record. These two models became icons—not least because of pop culture. From 1962 to 1995, James Bond famously wore the Submariner, cementing its badge of cool. Then, in *GoldenEye* (1995), Bond switched to the Seamaster, signaling a new era. Both watches remain benchmarks: the Submariner for its rugged elegance, the Seamaster for its blend of heritage and innovation.'

In [ ]:
template = PromptTemplate.from_template(
"""
    Persona: you are a watch enthusiast, experienced in writing high quality articles for watch magazines for a general audience\n
    instruction: {instruction},\n
    text: {text}\n
    tone: use a sober, knowledgeable but not too formal tone and use everyday vocabulary 
    Output: the length of the article should be 150 words at most
""")

instruction = """Write a piece on the diver watches"""
fact = """Facts to mention:
-the Rolex Oyster Perpetual was among the first to be water resistant
=Omega produced the Seamaster in 1948 as the first purpose designed diver watch was the Seamaster
=the Seamaster set a world record of immersion of 62.5m in 1955.
=Rolex produced the first Submariner in 1953
=the Submariner became popular also because of the association with James Bond until 1995
=James Bond started to wear the Seamaster in 1995"""

prompt = template.format(
    instruction=instruction,
    text=fact)

response = LLM.invoke(prompt)
display(response)

'Diver watches, like home PCs of the 1980s, are more than tools—they’re symbols of purpose-built ingenuity. Just as the Apple II or Commodore 64 were designed for real-world use—word processing, gaming, and exploration—the classic diver’s watch was engineered for survival underwater. Its key features—robust case, water resistance (typically 200m+), luminescent markers, unidirectional bezel—were not mere flourishes but life-safety necessities. Developed alongside the rise of saturation diving and naval operations, these watches earned trust through reliability, not aesthetics alone. Though modern smartwatches pack more features, the diver remains respected for its clarity, ruggedness, and timeless design language. Like those early PCs, diver watches represent a moment when technology met practical need, laying groundwork for everything that followed—even as their core mission stays unchanged: keep time, no matter the depth.'

#### 2.5 Question answering

In [ ]:
template = PromptTemplate.from_template(
"""
    instruction: read the following text, and tell me which home computers were popular in the 80s.\n
    text: {text}\n
    
""")

fact = """Home PCs from the 1980s were iconic pioneers of personal computing. Computers like the Apple II, IBM PC, and Commodore 64 revolutionized how we lived and worked. They introduced us to word processing, gaming, and the possibilities of home computing. The graphical user interface, as seen in the Apple Macintosh, changed the way we interacted with machines. These PCs, though less powerful by today's standards, ignited a digital revolution, shaping the technology landscape we know today. They hold a special place in history, symbolizing the dawn of a new era in computing that laid the foundation for the modern digital age."""

prompt = template.format(
    text=fact)

response = LLM.invoke(prompt)
display(response)

'Based on the provided text, the home computers popular in the 1980s mentioned are:\n\n- **Apple II**  \n- **IBM PC**  \n- **Commodore 64**  \n- **Apple Macintosh** (though the Macintosh was more influential for its GUI and launched in 1984, it was relatively expensive and less common as a *home* computer compared to the others listed—more so used in education and professional settings—but it is explicitly named and culturally iconic of the era).\n\nThe text highlights these as pivotal 1980s home PCs that helped drive the personal computing revolution.'

In [ ]:
template = PromptTemplate.from_template(
"""
    instruction: read the following text, and tell me which home computers were popular in the 80s.\n
    text: {text}\n
    Output: just output the names of the computers in a list
    
""")

fact = """Home PCs from the 1980s were iconic pioneers of personal computing. Computers like the Apple II, IBM PC, and Commodore 64 revolutionized how we lived and worked. They introduced us to word processing, gaming, and the possibilities of home computing. The graphical user interface, as seen in the Apple Macintosh, changed the way we interacted with machines. These PCs, though less powerful by today's standards, ignited a digital revolution, shaping the technology landscape we know today. They hold a special place in history, symbolizing the dawn of a new era in computing that laid the foundation for the modern digital age."""

prompt = template.format(
    text=fact)

response = LLM.invoke(prompt)
display(response)

'- Apple II  \n- IBM PC  \n- Commodore 64  \n- Apple Macintosh'

#### 2.5 Reasoning

In [37]:
template = PromptTemplate.from_template(
"""
    instruction: Add the square numbers in this sequence: 19, 13, 1, 17, 4, 64, 900\n
""")



prompt = template.format()

response = LLM.invoke(prompt)
display(response)

'We are asked to **add the square numbers** in the sequence:\n\n**19, 13, 1, 17, 4, 64, 900**\n\nFirst, identify which numbers in the sequence are **perfect squares**.\n\n- **19** → not a perfect square  \n- **13** → not a perfect square  \n- **1** → $1 = 1^2$ ✅  \n- **17** → not a perfect square  \n- **4** → $4 = 2^2$ ✅  \n- **64** → $64 = 8^2$ ✅  \n- **900** → $900 = 30^2$ ✅  \n\nSo the square numbers are: **1, 4, 64, 900**\n\nNow, add them:\n\n$$\n1 + 4 = 5 \\\\\n5 + 64 = 69 \\\\\n69 + 900 = 969\n$$\n\n### ✅ Final Answer:\n**969**'

In [38]:
template = PromptTemplate.from_template("""Sum the palindromes in this sequence: 13, 1331, 121, 73, 99, 56, 232, 7""")

prompt = template.format()

response = LLM.invoke(prompt)
display(response)

'We are given the sequence:\n\n**13, 1331, 121, 73, 99, 56, 232, 7**\n\nWe need to **sum the palindromes** in this sequence.\n\n---\n\n### Step 1: Understand what a palindrome is  \nA **palindrome** is a number (or word) that reads the same forwards and backwards.\n\nLet’s check each number:\n\n1. **13** → reverse is **31** → not a palindrome  \n2. **1331** → reverse is **1331** → ✅ palindrome  \n3. **121** → reverse is **121** → ✅ palindrome  \n4. **73** → reverse is **37** → not a palindrome  \n5. **99** → reverse is **99** → ✅ palindrome  \n6. **56** → reverse is **65** → not a palindrome  \n7. **232** → reverse is **232** → ✅ palindrome  \n8. **7** → single-digit numbers are always palindromes → ✅ palindrome  \n\nSo the palindromes are:\n\n**1331, 121, 99, 232, 7**\n\n---\n\n### Step 2: Sum them\n\nLet’s compute:\n\n- 1331 + 121 = 1452  \n- 1452 + 99 = 1551  \n- 1551 + 232 = 1783  \n- 1783 + 7 = **1790**\n\n---\n\n### ✅ Final Answer:\n**1790**'

### Reasoning in detail
#### 1. One-shot learning

In [41]:
template = PromptTemplate.from_template("""Sum the palindromes in this sequence: 13, 1331, 121, 73, 99, 56, 232, 7""")

prompt = template.format()

response = LLM.invoke(prompt)
display(response)

KeyboardInterrupt: 

#### 2. Two-shot learning

In [ ]:
response = LLM.invoke("""Instruction: Sum the palindromes in this sequence: 13, 1331, 121, 73, 99, 56, 232, 7
Examples:
33 is a palindrome
44 is a palindrome""")

display(response)

'We are given the sequence:\n\n**13, 1331, 121, 73, 99, 56, 232, 7**\n\nWe need to identify which of these numbers are **palindromes**, then sum them.\n\nA **palindrome** reads the same forwards and backwards.\n\nLet’s check each number:\n\n1. **13** → forwards: "13", backwards: "31" → ❌ not a palindrome  \n2. **1331** → "1331" reversed is "1331" → ✅ palindrome  \n3. **121** → "121" reversed is "121" → ✅ palindrome  \n4. **73** → "73" reversed is "37" → ❌  \n5. **99** → "99" reversed is "99" → ✅ palindrome  \n6. **56** → "56" reversed is "65" → ❌  \n7. **232** → "232" reversed is "232" → ✅ palindrome  \n8. **7** → single-digit numbers are always palindromes → ✅ palindrome  \n\nSo the palindromes are:  \n**1331, 121, 99, 232, 7**\n\nNow sum them:\n\n- 1331 + 121 = 1452  \n- 1452 + 99 = 1551  \n- 1551 + 232 = 1783  \n- 1783 + 7 = **1790**\n\n---\n\n✅ **Final Answer: 1790**'

#### 3. Providing steps

In [ ]:
response = LLM.invoke("""Instruction: Sum the palindromes in this sequence: 13, 1331, 121, 73, 99, 56 232, 7
Examples:
33 is a palindrome
44 is a palindrome
                      
Steps: 1) identify the palindromes; 2) add them up""")

display(response)

'Let\'s go step by step.\n\nWe are given the sequence:\n\n**13, 1331, 121, 73, 99, 56 232, 7**\n\nWait — there\'s a possible spacing issue: "**56 232**" — is that two numbers: **56** and **232**, or is it **56232**?  \nGiven the context and the examples (single-digit and multi-digit numbers like 33, 44), it\'s most likely **56** and **232** are separate numbers.\n\nSo the full list is:\n\n- 13  \n- 1331  \n- 121  \n- 73  \n- 99  \n- 56  \n- 232  \n- 7  \n\nNow, identify which of these are palindromes.\n\nA **palindrome** reads the same forwards and backwards.\n\n1. **13** → reverse is **31** → not a palindrome  \n2. **1331** → reverse is **1331** → ✅ palindrome  \n3. **121** → reverse is **121** → ✅ palindrome  \n4. **73** → reverse is **37** → ❌  \n5. **99** → reverse is **99** → ✅ palindrome  \n6. **56** → reverse is **65** → ❌  \n7. **232** → reverse is **232** → ✅ palindrome  \n8. **7** → single digit → always a palindrome → ✅\n\nSo the palindromes are:\n\n- 1331  \n- 121  \n- 99  

#### 4. Few-shot learning

In [42]:
response = LLM.invoke("""Classify the following numbers as Abra, Kadabra or Abra Kadabra:

3, 4, 5, 7, 8, 10, 11, 13, 35

Examples:

6 // not divisible by 5, not divisible by 7 // None

15 // divisible by 5, not divisible by 7 // Abra

12 // not divisible by 5, not divisible by 7 // None

21 // not divisible by 5, divisible by 7 // Kadabra

70 // divisible by 5, divisible by 7 // Abra Kadabra""")

display(response)

"Let's classify each number based on divisibility:\n\n- **Abra**: divisible by 5  \n- **Kadabra**: divisible by 7  \n- **Abra Kadabra**: divisible by both 5 **and** 7 (i.e., divisible by 35)  \n- **None**: divisible by neither\n\nLet’s go through each number:\n\n1. **3**:  \n   - 3 ÷ 5 = 0.6 → not divisible by 5  \n   - 3 ÷ 7 ≈ 0.428 → not divisible by 7  \n   → **None**\n\n2. **4**:  \n   - Not divisible by 5 or 7  \n   → **None**\n\n3. **5**:  \n   - 5 ÷ 5 = 1 → divisible by 5  \n   - 5 ÷ 7 ≈ 0.714 → not divisible by 7  \n   → **Abra**\n\n4. **7**:  \n   - 7 ÷ 5 = 1.4 → not divisible by 5  \n   - 7 ÷ 7 = 1 → divisible by 7  \n   → **Kadabra**\n\n5. **8**:  \n   - Not divisible by 5 or 7  \n   → **None**\n\n6. **10**:  \n   - 10 ÷ 5 = 2 → divisible by 5  \n   - 10 ÷ 7 ≈ 1.428 → not divisible by 7  \n   → **Abra**\n\n7. **11**:  \n   - Not divisible by 5 or 7  \n   → **None**\n\n8. **13**:  \n   - Not divisible by 5 or 7  \n   → **None**\n\n9. **35**:  \n   - 35 ÷ 5 = 7 → divisible by 

In [43]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
 
examples = [
  {
      "number": 6,
      "reasoning": "not divisible by 5 nor by 7",
      "result": "None"
  },
  {
      "number": 15,
      "reasoning": "divisible by 5 but not by 7",
      "result": "Abra"
  },
  {
      "number": 12,
      "reasoning": "not divisible by 5 nor by 7",
      "result": "None"
  },
  {
      "number": 21,
      "reasoning": "divisible by 7 but not by 5",
      "result": "Kadabra"
  },
  {
      "number": 70,
      "reasoning": "divisible by 5 and by 7",
      "result": "Abra Kadabra"
  } ]
 
example_prompt = PromptTemplate(input_variables=["number","reasoning", "result"], 
                                template="{number} \\ {reasoning} \\ {result}")
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="""Classify the following numbers as Abra, Kadabra
    or Abra Kadabra: {comma_delimited_input_numbers}""",
    input_variables=["comma_delimited_input_numbers"]
)
 
prompt_input = few_shot_prompt.format(comma_delimited_input_numbers="3, 4, 5, 7, 8, 10, 11, 13, 35.")
 
response = LLM.invoke(prompt_input)
print(response)

Let's review the pattern from the examples:

- **Divisible by 5 only** → **Abra**  
- **Divisible by 7 only** → **Kadabra**  
- **Divisible by both 5 and 7** (i.e., divisible by **35**) → **Abra Kadabra**  
- **Not divisible by 5 nor by 7** → **None**

Now classify each number:

1. **3**  
   - 3 ÷ 5 = 0.6 → not divisible  
   - 3 ÷ 7 ≈ 0.428 → not divisible  
   → **None**

2. **4**  
   - Not divisible by 5 or 7  
   → **None**

3. **5**  
   - Divisible by 5 (5 ÷ 5 = 1)  
   - Not divisible by 7 (5 ÷ 7 ≈ 0.714)  
   → **Abra**

4. **7**  
   - Divisible by 7 (7 ÷ 7 = 1)  
   - Not divisible by 5  
   → **Kadabra**

5. **8**  
   - Not divisible by 5 or 7  
   → **None**

6. **10**  
   - 10 ÷ 5 = 2 → divisible by 5  
   - 10 ÷ 7 ≈ 1.428 → not divisible by 7  
   → **Abra**

7. **11**  
   - Not divisible by 5 or 7  
   → **None**

8. **13**  
   - Not divisible by 5 or 7  
   → **None**

9. **35**  
   - 35 ÷ 5 = 7 → divisible by 5  
   - 35 ÷ 7 = 5 → divisible by 7  
   → **Abra Ka

#### 5. Chain of Thought (CoT)

In [44]:

 
 
response = LLM.invoke("""
                      Q: Is the following sequence "Strange"?
                      3, 4, 5, 7, 10, 18, 22, 24
                      A:
                      Examples:               
                      Q: is the following sequence strange: 1, 4, 6, 8, 20
                        A: 1 is an odd number; I need at least two odd numbers // Not Strange

Q: is the following sequence strange: 5, 6, 7, 8, 20

A: 5 and 7 are odd numbers; the sum of 5 and 7 is 12; 12 is divisible by 3 // Strange

Q: is the following sequence strange: 1, 5, 6, 7, 8, 20

A: 1, 5 and 7 are odd numbers; the sum of 1, 5 and 7 is 13; 13 is not divisible by 3 // Not Strange

Q: is the following sequence strange: 5, 6, 7, 8, 9, 20

A: 5, 7, 9 are odd numbers; the sum of 5, 7 and 9 is 21; 21 is divisible by 3 // Strange""")
print(response)

Let’s analyze the given sequence:  
**3, 4, 5, 7, 10, 18, 22, 24**

We need to determine if it is **“Strange”**, based on the examples provided.

### Observing the pattern from examples:

The examples suggest a rule for “Strange”:

- Look at the **odd numbers** in the sequence.
- Compute their **sum**.
- If the **sum is divisible by 3**, then the sequence is **Strange**.
- Otherwise, **Not Strange**.

Let’s verify with the examples:

1. **1, 4, 6, 8, 20**  
   Odd numbers: **1** → only one odd → “I need at least two odd numbers” → Not Strange  
   ✅ This suggests *at least two odds* required — but also note: sum = 1, not divisible by 3 anyway.

2. **5, 6, 7, 8, 20**  
   Odds: 5, 7 → sum = 12 → 12 ÷ 3 = 4 → divisible → **Strange** ✅

3. **1, 5, 6, 7, 8, 20**  
   Odds: 1, 5, 7 → sum = 13 → 13 mod 3 = 1 → not divisible → **Not Strange** ✅

4. **5, 6, 7, 8, 9, 20**  
   Odds: 5, 7, 9 → sum = 21 → 21 ÷ 3 = 7 → divisible → **Strange** ✅

So the rule appears to be:  
> A sequence is **Stran

## Prompt structure
Combining all the prompt elements from earlier sections results in the following generalized structure:

- Persona: Specify the role you want the language model (LLM) to embody. Whether a skilled marketing manager, an award-winning sci-fi author, or a proficient copy editor for a computing magazine, set the expectations high.
= Context: Provide detailed background information to help the LLM comprehend the context surrounding your request. Precision enhances the likelihood of aligning with your expectations.
= Instruction: Clearly define the action you want the LLM to take with your input text. Whether it's translation, writing, rewriting, improving, classifying, ordering, or finding, be explicit.
- Input: This refers to specific context details, which could take the form of a paragraph, a query, or a list of key points. You may choose to label it accordingly, such as "Text" or "Question.", etc
- Steps: Outline the processing steps the LLM should follow to generate the output.
- Tone: Specify the desired tone of the LLM's answer—formal, informal, witty, enthusiastic, sober, friendly, etc. Combinations are possible.
- Output Format: Optionally specify the desired output format. If not specified, the LLM assumes a text answer. However, you can request formats like a table, CSV file (with column names), JSON (with attribute names), or other structured formats.
- Examples: In cases where the LLM may lack sufficient training for certain tasks, especially in text classification and reasoning, provide mini-inline training through examples. Even a few examples significantly enhance answer quality.